In [1]:
from __future__ import annotations

import json
from collections import defaultdict
from pathlib import Path

import pandas as pd
from pymatgen.analysis.phase_diagram import PDEntry, PhaseDiagram
from pymatgen.core import Composition

In [2]:
# ── Configuration ─────────────────────────────────────────────────────────────
# Root of all fidelity experiment data.
# Paths are resolved relative to the repo root (one level above notebooks/).
REPO_ROOT = Path("..")
FIDELITY_BASE = REPO_ROOT / "notebooks" / "data" / "fidelity"
OUTPUT_PATH   = REPO_ROOT / "data" / "initial_family_prompts.json"

assert FIDELITY_BASE.exists(), f"Fidelity data not found at {FIDELITY_BASE.resolve()}"
print(f"Fidelity base : {FIDELITY_BASE.resolve()}")
print(f"Output path   : {OUTPUT_PATH.resolve()}")

Fidelity base : /Users/weiyong/Code/intern_codes/MADE/notebooks/data/fidelity
Output path   : /Users/weiyong/Code/intern_codes/MADE/data/initial_family_prompts.json


In [3]:
# ── Helper functions ───────────────────────────────────────────────────────────

def to_formula_string(comp: Composition) -> str:
    """Format a composition as alphabetically-sorted elements with explicit 1s.

    Matches the style in existing initial_family_prompts.json, e.g. 'Mg4Sn1Sr3'.
    """
    reduced = comp.reduced_composition
    elements_sorted = sorted(reduced.as_dict().keys())
    return "".join(f"{el}{int(round(reduced[el]))}" for el in elements_sorted)


def get_gt_stable_formulas(episode_data: dict) -> list[str]:
    """Compute GT convex hull from an episode JSON and return stable formulas.

    Excludes unary (pure element) entries.
    """
    raw_entries = episode_data.get("phase_diagram_gt", [])
    entries = [PDEntry.from_dict(e) for e in raw_entries]
    pd_gt = PhaseDiagram(entries)
    stable = []
    for entry in pd_gt.stable_entries:
        if len(entry.composition.elements) < 2:
            continue  # skip elemental references
        stable.append(to_formula_string(entry.composition))
    return sorted(set(stable))


def normalize_formula(formula_str: str) -> str:
    """Normalize any formula string to the canonical to_formula_string style."""
    try:
        return to_formula_string(Composition(formula_str))
    except Exception:
        return formula_str.strip().replace(" ", "")


def collect_proposed_formulas(system_dir: Path) -> set[str]:
    """Collect all normalized formulas proposed by the agent across all episodes."""
    proposed: set[str] = set()
    traj_dir = system_dir / "trajectories"
    if not traj_dir.exists():
        return proposed
    for ep_file in traj_dir.glob("episode_*.json"):
        with open(ep_file) as f:
            data = json.load(f)
        for step in data.get("trajectory", []):
            rf = step.get("reduced_formula", "")
            if rf:
                proposed.add(normalize_formula(rf))
    return proposed


def complexity_score(formula_str: str) -> float:
    """Lower = simpler (easier for LLMs).

    Score = arity * 10 + sum of reduced formula coefficients.
    Penalises both higher element count and larger stoichiometric numbers.
    Examples:
        Mg1Sn1   -> 2*10 + 2  = 22  (binary, very simple)
        Mg2Sn1   -> 2*10 + 3  = 23
        Mg4Sn1Sr3 -> 3*10 + 8 = 38  (ternary, complex)
    """
    try:
        comp = Composition(formula_str).reduced_composition
        arity = len(comp.elements)
        coeff_sum = sum(round(v) for v in comp.as_dict().values())
        return arity * 10 + coeff_sum
    except Exception:
        return 999.0


def find_all_run_dirs(base: Path) -> list[Path]:
    """Walk fidelity base and return all run directories that have a 'systems' subdir."""
    run_dirs = []
    for model_dir in sorted(base.iterdir()):
        if not model_dir.is_dir():
            continue
        for nary_dir in sorted(model_dir.iterdir()):
            if not nary_dir.is_dir():
                continue
            for ts_dir in sorted(nary_dir.iterdir()):
                if not ts_dir.is_dir():
                    continue
                for run_dir in sorted(ts_dir.iterdir()):
                    if run_dir.is_dir() and (run_dir / "systems").exists():
                        run_dirs.append(run_dir)
    return run_dirs

In [4]:
# ── Load GT stable formulas and aggregate proposed formulas across all runs ────

run_dirs = find_all_run_dirs(FIDELITY_BASE)
print(f"Found {len(run_dirs)} run directories")
for rd in run_dirs:
    print(f"  {rd.relative_to(FIDELITY_BASE)}")

Found 12 run directories
  ministral/systems_quaternary_n10_maxatoms20_intermetallic_smact/20260312-132153/llm_react_orchestrator_systems_quaternary_n10_maxatoms20_intermetallic_smact_10systems_50queries_100stabilitymeV
  ministral/systems_quinary_n10_maxatoms20_intermetallic_smact/20260312-132153/llm_react_orchestrator_systems_quinary_n10_maxatoms20_intermetallic_smact_10systems_50queries_100stabilitymeV
  ministral/systems_ternary_n10_maxatoms20_intermetallic_smact/20260312-132153/llm_react_orchestrator_systems_ternary_n10_maxatoms20_intermetallic_smact_10systems_50queries_100stabilitymeV
  qwen3-30b-instr_reflx/systems_quaternary_n10_maxatoms20_intermetallic_smact/20260312-131554/llm_react_orchestrator_systems_quaternary_n10_maxatoms20_intermetallic_smact_10systems_50queries_100stabilitymeV
  qwen3-30b-instr_reflx/systems_quinary_n10_maxatoms20_intermetallic_smact/20260312-131554/llm_react_orchestrator_systems_quinary_n10_maxatoms20_intermetallic_smact_10systems_50queries_100stabili

In [5]:
# Accumulate per-system data across all model runs and n-ary levels.
# GT stable is loaded once per system (same ground truth regardless of run).
# Proposed formulas are unioned across ALL runs/models for a more robust signal.

system_gt_stable: dict[str, list[str]] = {}   # system_id -> sorted list of GT stable formulas
system_proposed: dict[str, set[str]] = defaultdict(set)  # system_id -> union of proposed formulas

for run_dir in run_dirs:
    systems_dir = run_dir / "systems"
    for sys_dir in sorted(systems_dir.iterdir()):
        if not sys_dir.is_dir():
            continue
        system_id = sys_dir.name

        # Load GT stable formulas from episode_000 (same for all models/runs)
        if system_id not in system_gt_stable:
            ep0 = sys_dir / "trajectories" / "episode_000.json"
            if ep0.exists():
                with open(ep0) as f:
                    ep0_data = json.load(f)
                gt_stable = get_gt_stable_formulas(ep0_data)
                system_gt_stable[system_id] = gt_stable

        # Collect proposed formulas from this run
        system_proposed[system_id] |= collect_proposed_formulas(sys_dir)

print(f"Processed {len(system_gt_stable)} unique systems")

Processed 30 unique systems


In [6]:
# ── Classify each GT stable composition as easy or hard ───────────────────────

MIN_PER_CLASS = 3  # guarantee at least this many in each class

results: dict[str, dict[str, list[str]]] = {}
fallback_systems: list[str] = []  # systems that needed the complexity fallback

for system_id in sorted(system_gt_stable):
    gt_set = set(system_gt_stable[system_id])
    proposed = system_proposed[system_id]

    # Stage 1: agent-based classification
    easy = sorted(gt_set & proposed)
    hard = sorted(gt_set - proposed)

    # Stage 2: complexity-based fallback if either class is too small
    if len(easy) < MIN_PER_CLASS or len(hard) < MIN_PER_CLASS:
        fallback_systems.append(system_id)

        # Sort ALL GT stable formulas by complexity (ascending = simpler = easy)
        all_stable = sorted(gt_set, key=complexity_score)
        n = len(all_stable)

        if n < 2 * MIN_PER_CLASS:
            # Not enough total GT compounds to guarantee MIN_PER_CLASS in both;
            # split as evenly as possible (easy gets the simpler half).
            split = n // 2
        else:
            # Reserve MIN_PER_CLASS at each end; split remainder at midpoint.
            split = max(MIN_PER_CLASS, n // 2)

        easy = sorted(all_stable[:split])
        hard = sorted(all_stable[split:])

    results[system_id] = {"hard": hard, "easy": easy}

print(f"Classification complete for {len(results)} systems")
if fallback_systems:
    print(f"  Complexity fallback used for {len(fallback_systems)} system(s):")
    for sid in fallback_systems:
        v = results[sid]
        print(f"    {sid}: easy={len(v['easy'])}, hard={len(v['hard'])}")
else:
    print("  All systems had ≥ MIN_PER_CLASS from agent data alone.")

Classification complete for 30 systems
  Complexity fallback used for 27 system(s):
    Ag-Nd-Pd-Pt-Tb: easy=13, hard=13
    Al-Hg-K-Mg-W: easy=4, hard=5
    Al-Li-V: easy=3, hard=4
    Al-Lu-Pt-Rb-Sm: easy=11, hard=12
    Al-V-Zn: easy=2, hard=3
    Au-Cr-Cs-Dy: easy=2, hard=3
    Au-K-Tb: easy=3, hard=3
    Au-Tb-V-Y: easy=4, hard=5
    Ba-Be-Hf-Li: easy=1, hard=1
    Ba-Nd-Ni-W: easy=3, hard=3
    Ca-Fe-Gd-Pb-Tb: easy=5, hard=5
    Cd-Gd-Mn-Na-Ta: easy=3, hard=4
    Cd-Li-Nd-Ti-W: easy=4, hard=4
    Ce-Er-Pb-Rh: easy=8, hard=9
    Ce-Ir-Pt-Sn: easy=11, hard=11
    Co-Dy-Ta-Y: easy=5, hard=6
    Co-Dy-W: easy=2, hard=3
    Co-Hg-Mg-Sr-W: easy=5, hard=5
    Co-Mg-Na: easy=0, hard=1
    Co-Pd-Tl: easy=1, hard=2
    Cr-Fe-Lu-Pt-Sc: easy=6, hard=6
    Dy-K-Pd-Sm: easy=3, hard=4
    Eu-Nb-Sn-Tl: easy=5, hard=5
    Ga-Ho-Lu: easy=4, hard=4
    Ga-Pt-Tm: easy=7, hard=7
    Hf-Ni-Zr: easy=3, hard=3
    Mg-Sn-Sr: easy=4, hard=4


In [7]:
# ── Summary table ──────────────────────────────────────────────────────────────

rows = []
for sid, v in sorted(results.items()):
    n_easy = len(v["easy"])
    n_hard = len(v["hard"])
    n_total = n_easy + n_hard
    arity = len(sid.split("-"))
    rows.append({
        "system": sid,
        "arity": arity,
        "gt_stable": n_total,
        "easy": n_easy,
        "hard": n_hard,
        "easy_%": f"{100 * n_easy / max(n_total, 1):.0f}%",
        "easy_formulas": ", ".join(v["easy"]),
        "hard_formulas": ", ".join(v["hard"]),
    })

df = pd.DataFrame(rows)

# Brief summary per n-ary level
print("── Per-arity summary ───────────────────────────────")
print(
    df.groupby("arity")[["gt_stable", "easy", "hard"]]
    .mean()
    .round(1)
    .rename(columns={"gt_stable": "avg_gt_stable", "easy": "avg_easy", "hard": "avg_hard"})
)

print("\n── Per-system breakdown ────────────────────────────")
pd.set_option("display.max_colwidth", 120)
pd.set_option("display.max_rows", 60)
df[["system", "arity", "gt_stable", "easy", "hard", "easy_%"]]

── Per-arity summary ───────────────────────────────
       avg_gt_stable  avg_easy  avg_hard
arity                                   
3                6.3       2.9       3.4
4               10.6       5.6       5.0
5               16.6       9.8       6.8

── Per-system breakdown ────────────────────────────


,system,arity,gt_stable,easy,hard,easy_%
0,Ag-Nd-Pd-Pt-Tb,5,26,13,13,50%
1,Al-Hg-K-Mg-W,5,9,4,5,44%
2,Al-Li-V,3,7,3,4,43%
3,Al-Lu-Pt-Rb-Sm,5,23,11,12,48%
4,Al-V-Zn,3,5,2,3,40%
5,Au-Cr-Cs-Dy,4,5,2,3,40%
6,Au-K-Tb,3,6,3,3,50%
7,Au-Tb-V-Y,4,9,4,5,44%
8,Ba-Be-Hf-Li,4,2,1,1,50%
9,Ba-Nd-Ni-W,4,6,3,3,50%


In [8]:
# ── Inspect specific system ────────────────────────────────────────────────────
# Change INSPECT_SYSTEM to any system_id you want to examine.

INSPECT_SYSTEM = "Mg-Sn-Sr"

if INSPECT_SYSTEM in results:
    v = results[INSPECT_SYSTEM]
    print(f"System: {INSPECT_SYSTEM}")
    print(f"  GT stable total : {len(v['easy']) + len(v['hard'])}")
    print(f"  Easy ({len(v['easy'])}): {v['easy']}")
    print(f"  Hard ({len(v['hard'])}): {v['hard']}")
else:
    print(f"{INSPECT_SYSTEM} not found. Available: {sorted(results.keys())}")

System: Mg-Sn-Sr
  GT stable total : 8
  Easy (4): ['Mg2Sn1', 'Mg2Sr1', 'Sn1Sr1', 'Sn1Sr2']
  Hard (4): ['Mg1Sn1Sr1', 'Sn3Sr1', 'Sn3Sr5', 'Sn5Sr3']


In [9]:
# ── Verify minimum per-class guarantee ────────────────────────────────────────

under_min = [
    (sid, len(v["easy"]), len(v["hard"]))
    for sid, v in results.items()
    if len(v["easy"]) < MIN_PER_CLASS or len(v["hard"]) < MIN_PER_CLASS
]
if under_min:
    print(f"WARNING: {len(under_min)} system(s) still below MIN_PER_CLASS={MIN_PER_CLASS}:")
    for sid, n_easy, n_hard in sorted(under_min):
        print(f"  {sid}: easy={n_easy}, hard={n_hard}  (GT stable total = {n_easy + n_hard})")
    print("These systems simply don't have enough GT stable compounds to fill both classes.")
else:
    print(f"All {len(results)} systems have ≥ {MIN_PER_CLASS} easy AND ≥ {MIN_PER_CLASS} hard compositions.")

  Al-V-Zn: easy=2, hard=3  (GT stable total = 5)
  Au-Cr-Cs-Dy: easy=2, hard=3  (GT stable total = 5)
  Ba-Be-Hf-Li: easy=1, hard=1  (GT stable total = 2)
  Co-Dy-W: easy=2, hard=3  (GT stable total = 5)
  Co-Mg-Na: easy=0, hard=1  (GT stable total = 1)
  Co-Pd-Tl: easy=1, hard=2  (GT stable total = 3)
These systems simply don't have enough GT stable compounds to fill both classes.


In [10]:
# ── Export to data/initial_family_prompts.json ────────────────────────────────

with open(OUTPUT_PATH, "w") as f:
    json.dump(results, f, indent=2)

print(f"Written {len(results)} systems to {OUTPUT_PATH.resolve()}")

# Quick sanity-check: print first 3 systems from the output
with open(OUTPUT_PATH) as f:
    loaded = json.load(f)
for sid in list(loaded)[:3]:
    print(f"\n{sid}:")
    print(f"  hard ({len(loaded[sid]['hard'])}): {loaded[sid]['hard'][:5]} ...")
    print(f"  easy ({len(loaded[sid]['easy'])}): {loaded[sid]['easy'][:5]} ...")

Written 30 systems to /Users/weiyong/Code/intern_codes/MADE/data/initial_family_prompts.json

Ag-Nd-Pd-Pt-Tb:
  hard (13): ['Ag1Pt4', 'Ag3Pd1', 'Nd1Pd3', 'Nd1Pt5', 'Nd3Pd4'] ...
  easy (13): ['Ag1Nd1', 'Ag1Pd1', 'Ag1Pt1', 'Ag1Tb1', 'Ag2Nd1'] ...

Al-Hg-K-Mg-W:
  hard (5): ['Al12W1', 'Al5W1', 'Hg1Mg3', 'Hg3Mg5', 'Hg7K2'] ...
  easy (4): ['Al2Mg1', 'Hg1K1', 'Hg1Mg1', 'Hg2K1'] ...

Al-Li-V:
  hard (4): ['Al1V3', 'Al2Li3', 'Al3Li1', 'Al3V1'] ...
  easy (3): ['Al1Li1', 'Al1Li2', 'Al1V1'] ...
